# Paper §3.1: Taylor–Green vortex and Poiseuille flow (accuracy)

This notebook reproduces the grid-convergence figure of Section 3.1 of the paper *GeLB: A Domain-Specific Language and Compiler for Lattice Boltzmann Simulations* (Chirila & Lohmann, submitted to SoftwareX, 2026), with the GeLB programs the paper used ([`examples/softwarex-2026-paper/`](https://github.com/dchirila/gelb/tree/v1.0.0/examples/softwarex-2026-paper), release v1.0.0).

Three flows with analytical solutions are run at increasing resolution, and the $L_2$ error against the analytical solution is plotted against the grid size: the Taylor–Green vortex with BGK and with MRT collision, and body-force-driven Poiseuille flow with MRT collision. All three converge at second order, as the lattice Boltzmann method should.

The notebook runs on a free Colab CPU runtime. Expected runtime: **5–10 minutes** (3 minutes on a laptop with a Ryzen 9 7945HX), most of it the Poiseuille runs, each of which lasts five viscous diffusion times.

## 1. Install the compilers' runtime dependencies

The Fortran code that `gelbc` emits needs `gfortran` and the NetCDF-Fortran library, which writes the gauge output.

In [ ]:
!apt-get update -qq
!apt-get install -y -qq gfortran libnetcdff-dev > /dev/null
!gfortran --version | head -1

## 2. Install GeLB

GeLB 1.0.0, the release described in the paper, installs from its tag on GitHub (about 30 s). The CPU runs use the Fortran backend (`-b cpu_fortran_serial`): a pip installation has no Highway library, which the default `cpu_highway` backend needs (see the [README](https://github.com/dchirila/gelb#installation)). The backends compute the same results; the paper's timings used `cpu_highway`.

In [ ]:
!pip install -q "git+https://github.com/dchirila/gelb.git@v1.0.0"

# gelb-run starts the compiler and its progress monitor through a Python
# interpreter that can import gelb: name this notebook's own.
import os, sys
os.environ["GELB_PYTHON"] = sys.executable

!python -c "import importlib.metadata as m; print('gelb', m.version('gelb'))"

GeLB source is shown with syntax highlighting by the Pygments lexer that the GeLB manuals use:

In [ ]:
!curl -sSfL -o gelb_lexer.py https://raw.githubusercontent.com/dchirila/gelb/v1.0.0/docs/_ext/gelb_lexer.py

from gelb_lexer import GelbLexer
from pygments import highlight
from pygments.formatters import HtmlFormatter
from IPython.display import HTML, display

def show_gelb(path, max_height=None):
    """Display a .gelb file with syntax highlighting and line numbers."""
    with open(path) as f:
        html = highlight(f.read(), GelbLexer(),
                         HtmlFormatter(noclasses=True, style="friendly", linenos="inline",
                                       prestyles="font-family: monospace; font-size: 13px; line-height: 125%;"))
    box = f"max-height: {max_height}px; overflow: auto;" if max_height else ""
    display(HTML(f'<div style="{box}">{html}</div>'))

## 3. Get the paper's programs

In [ ]:
BASE = "https://raw.githubusercontent.com/dchirila/gelb/v1.0.0/examples/softwarex-2026-paper"
for name in ["taylor_green_vortex_2d_bgk.gelb", "taylor_green_vortex_2d_mrt.gelb", "poiseuille_flow_2d_mrt.gelb"]:
    !curl -sSfL -o {name} {BASE}/{name}
!ls -l *.gelb

The BGK Taylor–Green program is the complete program listed in the paper's appendix:

In [ ]:
show_gelb('taylor_green_vortex_2d_bgk.gelb')

## 4. Run the resolution sweeps

`-D NAME=VALUE` overrides a global constant at compile time, so one program serves every resolution.

- **Taylor–Green**: each run lasts one velocity e-folding time, $t = 1/(2\nu k^2)$ with $k = 2\pi/N$; the error is taken over the whole velocity field against $u = U\,e^{-2\nu k^2 t}(-\cos kx \sin ky,\ \sin kx \cos ky)$.
- **Poiseuille**: each run lasts five viscous diffusion times $H^2/\nu$, long enough to reach the steady parabola $u(y) = \frac{g}{2\nu}\,y\,(H-y)$; the walls sit half a cell inside the wall rows (half-way bounce-back). The flow does not vary along the channel, so the notebook runs a channel 4 cells long instead of the program's 128 (`-D NX=4`): the error is the same to the last digit printed, and the runs are 30 times shorter.

In [ ]:
import subprocess

def gelb_run(program, steps, write_every, backend="cpu_fortran_serial", **defines):
    """Compile and run a GeLB program quietly; return the NetCDF file it wrote."""
    cmd = ["gelb-run", "-b", backend, "-n", str(steps), "-w", str(write_every)]
    for name, value in defines.items():
        cmd += ["-D", f"{name}={value}"]
    p = subprocess.run(cmd + [program], capture_output=True, text=True)
    if p.returncode:
        raise RuntimeError(p.stdout[-2000:] + p.stderr[-2000:])
    return program.replace(".gelb", ".nc")

In [ ]:
import netCDF4
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# The paper's palette (Okabe-Ito): blue = read / cold, vermillion = write / warm.
BLUE, VERMILLION, GREEN, GRAY, INK = "#0072B2", "#D55E00", "#009E73", "#878787", "#333333"
DIVERGING = LinearSegmentedColormap.from_list("gelb_div", [BLUE, "#f4f3f1", VERMILLION])

def last_record(nc, *names):
    """The last gauge record of each variable, with non-fluid cells as NaN."""
    with netCDF4.Dataset(nc) as ds:
        return [np.ma.filled(ds.variables[n][-1], np.nan) for n in names]

In [ ]:
U_MAX = 0.04
TGV = {"Taylor–Green, BGK": ("taylor_green_vortex_2d_bgk.gelb", 1.0 / 6.0),
       "Taylor–Green, MRT": ("taylor_green_vortex_2d_mrt.gelb", 1.0 / 30.0)}

def tgv_error(program, nu, N):
    k = 2.0 * np.pi / N
    steps = int(1.0 / (2.0 * nu * k * k))
    ux, uy = last_record(gelb_run(program, steps, steps, N=N), "ux", "uy")
    xc = np.arange(N) + 0.5
    X, Y = np.meshgrid(xc, xc)
    decay = np.exp(-2.0 * nu * k * k * steps)
    ux_a = -U_MAX * np.cos(k * X) * np.sin(k * Y) * decay
    uy_a = U_MAX * np.sin(k * X) * np.cos(k * Y) * decay
    return np.sqrt(((ux - ux_a)**2 + (uy - uy_a)**2).sum() / (ux_a**2 + uy_a**2).sum())

results = {}
for label, (program, nu) in TGV.items():
    results[label] = ([16, 32, 64, 128], [])
    for N in results[label][0]:
        results[label][1].append(tgv_error(program, nu, N))
        print(f"{label}  N = {N:4d}  L2 = {results[label][1][-1]:.2e}")

The Poiseuille runs up to $N_y = 128$ take a few minutes; $N_y = 128$ alone is six million steps. The paper's figure goes to $N_y = 256$, eight times the work of $N_y = 128$; set `INCLUDE_256 = True` to run it too (about 17 minutes more on the laptop above, longer on Colab).

In [ ]:
INCLUDE_256 = False

TAU = 0.54                    # the program's relaxation time
NU_P = (TAU - 0.5) / 3.0
G_BASE, NY_BASE = 1e-5, 32    # the program scales its body force as G_BASE * (NY_BASE-2)^2 / (NY-2)^2

def poiseuille_error(NY):
    H = NY - 2
    steps = max(5 * int(H * H / NU_P), 10000)
    (ux,) = last_record(gelb_run("poiseuille_flow_2d_mrt.gelb", steps, steps, NY=NY, NX=4), "ux")
    u_sim = ux[1:NY - 1, 2]
    g = G_BASE * (NY_BASE - 2)**2 / H**2
    y = np.arange(1, NY - 1) - 0.5
    u_a = g / (2.0 * NU_P) * y * (H - y)
    return np.sqrt(((u_sim - u_a)**2).sum() / (u_a**2).sum())

label = "Poiseuille, MRT"
results[label] = ([16, 32, 64, 128] + ([256] if INCLUDE_256 else []), [])
for NY in results[label][0]:
    results[label][1].append(poiseuille_error(NY))
    print(f"{label}  NY = {NY:4d}  L2 = {results[label][1][-1]:.2e}")

## 5. Compare with the paper

Open markers are the values plotted in the paper (from [`examples/VALIDATION.md`](https://github.com/dchirila/gelb/blob/v1.0.0/examples/VALIDATION.md)), filled markers this notebook's runs.

In [ ]:
PAPER = {"Taylor–Green, BGK": ([16, 32, 64, 128], [1.3e-2, 2.9e-3, 9.9e-4, 4.5e-4]),
         "Taylor–Green, MRT": ([16, 32, 64, 128], [2.3e-2, 5.4e-3, 1.0e-3, 2.8e-4]),
         "Poiseuille, MRT":   ([16, 32, 64, 128, 256], [6.3e-3, 1.4e-3, 3.2e-4, 7.8e-5, 1.9e-5])}
STYLE_OF = {"Taylor–Green, BGK": ("o", BLUE), "Taylor–Green, MRT": ("s", VERMILLION),
            "Poiseuille, MRT": ("^", GREEN)}

print(f"{'case':20s} {'N':>5s} {'this run':>10s} {'paper':>10s}")
for label, (ns, errs) in results.items():
    paper = dict(zip(*PAPER[label]))
    for n, e in zip(ns, errs):
        print(f"{label:20s} {n:5d} {e:10.2e} {paper[n]:10.1e}")

fig, ax = plt.subplots(figsize=(6, 4))
for label, (ns, errs) in results.items():
    marker, color = STYLE_OF[label]
    ax.loglog(ns, errs, marker + "-", color=color, lw=1.6, ms=6, label=label)
    ax.loglog(*PAPER[label], marker, color=color, ms=11, mfc="none", mew=1.0)
n_ref = np.array([16, 256])
ax.loglog(n_ref, 2.2e-3 * (n_ref / 16.0)**-2, "--", color=GRAY, lw=1.2, label="slope $-2$")
ax.minorticks_off()
ax.set_xticks([16, 32, 64, 128, 256]); ax.set_xticklabels(["16", "32", "64", "128", "256"])
ax.set_xlabel("grid resolution $N$"); ax.set_ylabel("$L_2$ error vs. analytical solution")
ax.grid(True, which="major", color="#dddddd", lw=0.5)
ax.legend(frameon=False, loc="lower left")
plt.tight_layout(); plt.show()

The paper's Section 3.1 also reports CPU throughput (MLUPS) and the comparison with lbmpy. Those numbers depend on the machine, so they are not reproduced here; [`docs/benchmarks/lbmpy_reproduction.md`](https://github.com/dchirila/gelb/blob/v1.0.0/docs/benchmarks/lbmpy_reproduction.md) has the recipe for a local checkout.

## Where to go from here

- The programs and the commands that produced the paper's figures: [`examples/softwarex-2026-paper/`](https://github.com/dchirila/gelb/tree/v1.0.0/examples/softwarex-2026-paper).
- The [GeLB User Tutorial](https://github.com/dchirila/gelb/blob/v1.0.0/docs/pdf/GeLB_User_Tutorial.pdf) (PDF) and the [repository](https://github.com/dchirila/gelb).
- Release 1.0.0 is archived as [doi:10.5281/zenodo.23592922](https://doi.org/10.5281/zenodo.23592922); the [README](https://github.com/dchirila/gelb#using-gelb-in-your-work) says how to cite GeLB.